In [ ]:


# Install Dependencies
!pip install roboflow ultralytics torch torchvision scikit-learn matplotlib seaborn opencv-python joblib --quiet

# Import Libraries
import numpy as np
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from ultralytics import YOLO
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, hamming_loss, f1_score, multilabel_confusion_matrix, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import cv2
import joblib

# Configuration
NUM_CLASSES = 6
DATASET_DIR = Path("/content/Traditional-boats-of-Bangladesh-16")
SEG_DATASET_DIR = Path("/content/Traditional-boats-of-Bangladesh-2-2")
TEST_IMAGES_DIR = DATASET_DIR / "valid/images"
TEST_LABELS_DIR = DATASET_DIR / "valid/labels"
NEW_IMAGES_DIR = Path("/content/test_images")
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Using device: {device}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 66.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 124.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 94.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 61.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from roboflow import Roboflow

# Object detection dataset
rf = Roboflow(api_key="na3kztUanfY3uHtBBIBA")
project = rf.workspace("dip-abbd6").project("traditional-boats-of-bangladesh")
dataset = project.version(16).download("yolov8", overwrite=True)

# Segmentation dataset
project_seg = rf.workspace("dip-abbd6").project("traditional-boats-of-bangladesh-2")
dataset_seg = project_seg.version(2).download("yolov8", overwrite=True)

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Traditional-boats-of-Bangladesh-16 in yolov8:: 100%|██████████| 7708/7708 [00:02<00:00, 2569.76it/s]


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Traditional-boats-of-Bangladesh-2-2 in yolov8:: 100%|██████████| 6702/6702 [00:02<00:00, 3272.31it/s]


In [ ]:
# Train YOLOv8 Detection Model
#yolo_model = YOLO("yolov8n.pt")
#yolo_model.train(data=str(DATASET_DIR / "data.yaml"), epochs=10, imgsz=640)



In [ ]:
# Train YOLOv8 Segmentation Model
#seg_model = YOLO("yolov8n-seg.pt")
#seg_model.train(data=str(SEG_DATASET_DIR / "data.yaml"), epochs=10, imgsz=640)

In [ ]:
#using pre-train models when not training

yolo_model = YOLO("/content/best.pt")
seg_model = YOLO("/content/best (4).pt")

In [ ]:
# Load ResNet18
resnet_model = models.resnet18(pretrained=True)
resnet_model.fc = nn.Linear(resnet_model.fc.in_features, NUM_CLASSES)
resnet_model = resnet_model.to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(resnet_model.parameters(), lr=1e-4)

# DataLoader for ResNet
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

class MultiLabelDataset(Dataset):
    def __init__(self, image_dir, label_dir, transform=None, num_classes=NUM_CLASSES):
        self.image_paths = sorted(Path(image_dir).glob("*.jpg"))
        self.label_dir = Path(label_dir)
        self.transform = transform
        self.num_classes = num_classes

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        image = Image.open(image_path).convert("RGB")
        label_path = self.label_dir / (image_path.stem + ".txt")
        label = np.zeros(self.num_classes)
        if label_path.exists():
            with open(label_path, "r") as f:
                for line in f:
                    cls = int(line.split()[0])
                    label[cls] = 1
        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(label, dtype=torch.float32)

    def __len__(self):
        return len(self.image_paths)

train_dataset = MultiLabelDataset(
    DATASET_DIR / "train/images",
    DATASET_DIR / "train/labels",
    transform
)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)



The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:00<00:00, 174MB/s]


In [ ]:
# #Train ResNet
# for epoch in range(10):
#     resnet_model.train()
#     running_loss = 0.0
#     for images, labels in train_loader:
#         images, labels = images.to(device), labels.to(device)
#         optimizer.zero_grad()
#         outputs = torch.sigmoid(resnet_model(images))
#         loss = criterion(outputs, labels)
#         loss.backward()
#         optimizer.step()
#         running_loss += loss.item()
#     print(f"Epoch {epoch+1}/10, Loss: {running_loss/len(train_loader):.4f}")

# # Save ResNet
# torch.save(resnet_model.state_dict(), "resnet18_multilabel.pt")

In [ ]:
#resnet18 pre-train
resnet_model.load_state_dict(torch.load("/content/resnet18_multilabel.pt"))

<All keys matched successfully>

In [ ]:
# --- Feature Extraction Functions ---
def extract_yolo_features(image_path):
    results = yolo_model.predict(source=str(image_path), save=False)
    result = results[0]
    bboxes = result.boxes
    num_boxes = len(bboxes)
    class_counts = np.zeros(NUM_CLASSES)
    confs = []
    for box in bboxes:
        cls = int(box.cls[0])
        conf = float(box.conf[0])
        class_counts[cls] += 1
        confs.append(conf)
    avg_conf = np.mean(confs) if confs else 0
    return np.concatenate([[num_boxes, avg_conf], class_counts]), bboxes


def extract_resnet_probs_crop(image, crop_box, img_size=224):
    x1, y1, x2, y2 = map(int, crop_box.xyxy[0])
    img_w, img_h = image.size
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(img_w, x2), min(img_h, y2)
    if x2 <= x1 or y2 <= y1:
        return np.zeros(NUM_CLASSES)
    crop = image.crop((x1, y1, x2, y2))
    tensor = transform(crop).unsqueeze(0).to(device)
    resnet_model.eval()
    with torch.no_grad():
        logits = resnet_model(tensor).cpu().squeeze(0).numpy()
        probs = 1 / (1 + np.exp(-logits))  # Sigmoid
    return probs


def extract_yolo_seg_features_crop(image, crop_box):
    x1, y1, x2, y2 = map(int, crop_box.xyxy[0])
    img_w, img_h = image.size
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(img_w, x2), min(img_h, y2)
    if x2 <= x1 or y2 <= y1:
        return np.array([0, 0])
    crop = image.crop((x1, y1, x2, y2))
    crop_path = "/tmp/crop.jpg"
    crop.save(crop_path)
    results = seg_model.predict(source=crop_path, save=False)
    result = results[0]
    areas, elongations = [], []
    if result.masks is not None:
        for mask in result.masks.data.cpu().numpy():
            binary_mask = (mask * 255).astype(np.uint8)
            contours, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            if contours:
                cnt = max(contours, key=cv2.contourArea)
                area = cv2.contourArea(cnt)
                x, y, w, h = cv2.boundingRect(cnt)
                elongation = float(w) / h if h > 0 else 0
                areas.append(area)
                elongations.append(elongation)
    avg_area = np.mean(areas) if areas else 0
    avg_elongation = np.mean(elongations) if elongations else 0
    return np.array([avg_area, avg_elongation])


def build_meta_feature(image_path):
    image = Image.open(image_path).convert('RGB')
    yolo_feat, bboxes = extract_yolo_features(image_path)
    resnet_probs_list = []
    seg_features_list = []
    for box in bboxes[:3]:  # limit to 3 crops
        resnet_probs = extract_resnet_probs_crop(image, box)
        resnet_probs_list.append(resnet_probs)
        seg_features = extract_yolo_seg_features_crop(image, box)
        seg_features_list.append(seg_features)
    resnet_feat = np.mean(resnet_probs_list, axis=0) if resnet_probs_list else np.zeros(NUM_CLASSES)
    seg_feat = np.mean(seg_features_list, axis=0) if seg_features_list else np.array([0, 0])
    return np.concatenate([yolo_feat, resnet_feat, seg_feat])


def get_multilabel_vector(label_path):
    label = np.zeros(NUM_CLASSES)
    if label_path.exists():
        with open(label_path, "r") as f:
            for line in f:
                cls = int(line.split()[0])
                label[cls] = 1
    return label


In [ ]:
# Feature Extraction Functions
def extract_yolo_features(image_path):
    # Run YOLOv8 detection to get bounding boxes, confidence, and class predictions
    results = yolo_model.predict(source=str(image_path), save=False)
    result = results[0]
    bboxes = result.boxes
    num_boxes = len(bboxes)  # Feature 1: Number of detected boats
    class_counts = np.zeros(NUM_CLASSES)  # Initialize counts for 6 classes
    confs = []  # Store confidence scores
    for box in bboxes:
        cls = int(box.cls[0])  # Class ID (0-5)
        conf = float(box.conf[0])  # Confidence score
        class_counts[cls] += 1  # Count detections per class
        confs.append(conf)
    avg_conf = np.mean(confs) if confs else 0  # Feature 2: Average confidence (0 if no boxes)
    return np.concatenate([[num_boxes, avg_conf], class_counts]), bboxes  # Return 8 features and boxes

def extract_resnet_probs_crop(image, crop_box, img_size=224):
    # Crop image using bounding box and prepare for ResNet
    x1, y1, x2, y2 = map(int, crop_box.xyxy[0])  # Extract box coordinates
    img_w, img_h = image.size
    # Ensure crop is within image bounds
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(img_w, x2), min(img_h, y2)
    if x2 <= x1 or y2 <= y1:  # Invalid crop
        return np.zeros(NUM_CLASSES)
    # Crop and transform image
    crop = image.crop((x1, y1, x2, y2))
    tensor = transform(crop).unsqueeze(0).to(device)
    resnet_model.eval()
    with torch.no_grad():
        logits = resnet_model(tensor).cpu().squeeze(0).numpy()
        probs = 1 / (1 + np.exp(-logits))  # Sigmoid to get probabilities
    return probs  # Return 6 class probabilities for the crop

def extract_yolo_seg_features_crop(image, crop_box):
    # Crop image and run YOLOv8 segmentation on the cropped region
    x1, y1, x2, y2 = map(int, crop_box.xyxy[0])
    img_w, img_h = image.size
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(img_w, x2), min(img_h, y2)
    if x2 <= x1 or y2 <= y1:  # Invalid crop
        return np.array([0, 0])
    crop = image.crop((x1, y1, x2, y2))
    # Save crop temporarily for YOLO segmentation
    crop_path = "/tmp/crop.jpg"
    crop.save(crop_path)
    results = seg_model.predict(source=crop_path, save=False)
    result = results[0]
    areas, elongations = [], []
    if result.masks is not None:
        for mask in result.masks.data.cpu().numpy():
            binary_mask = (mask * 255).astype(np.uint8)
            contours, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            if contours:
                cnt = max(contours, key=cv2.contourArea)
                area = cv2.contourArea(cnt)
                x, y, w, h = cv2.boundingRect(cnt)
                elongation = float(w) / h if h > 0 else 0
                areas.append(area)
                elongations.append(elongation)
    avg_area = np.mean(areas) if areas else 0
    avg_elongation = np.mean(elongations) if elongations else 0
    return np.array([avg_area, avg_elongation])  # Return 2 features

def build_meta_feature(image_path):
    # Load image
    image = Image.open(image_path).convert('RGB')
    # Extract YOLO detection features and bounding boxes
    yolo_feat, bboxes = extract_yolo_features(image_path)
    # Initialize lists for ResNet and segmentation features
    resnet_probs_list = []
    seg_features_list = []
    # Process each detected bounding box
    for box in bboxes:
        # Extract ResNet probabilities for the cropped region
        resnet_probs = extract_resnet_probs_crop(image, box)
        resnet_probs_list.append(resnet_probs)
        # Extract segmentation features for the cropped region
        seg_features = extract_yolo_seg_features_crop(image, box)
        seg_features_list.append(seg_features)
    # Aggregate features across crops (average if multiple, zeros if none)
    resnet_feat = np.mean(resnet_probs_list, axis=0) if resnet_probs_list else np.zeros(NUM_CLASSES)
    seg_feat = np.mean(seg_features_list, axis=0) if seg_features_list else np.array([0, 0])
    # Combine into 16-dimensional feature vector
    return np.concatenate([yolo_feat, resnet_feat, seg_feat])

# Build Meta-Classifier Dataset
def get_multilabel_vector(label_path):
    label = np.zeros(NUM_CLASSES)
    if label_path.exists():
        with open(label_path, "r") as f:
            for line in f:
                cls = int(line.split()[0])
                label[cls] = 1
    return label

image_paths = sorted((DATASET_DIR / "train/images").glob("*.jpg"))
label_dir = DATASET_DIR / "train/labels"
X, y = [], []

for img_path in image_paths:
    label_path = label_dir / (img_path.stem + ".txt")
    if not label_path.exists():
        print(f"Skipping {img_path.name}: Missing label file")
        continue
    label_vec = get_multilabel_vector(label_path)
    feat_vec = build_meta_feature(img_path)
    if feat_vec.size == 0:
        print(f"Skipping {img_path.name}: Empty feature vector")
        continue
    X.append(feat_vec)
    y.append(label_vec)

X, y = np.array(X), np.array(y)
print(f"Total samples: {len(X)}, Feature shape: {X.shape}, Label shape: {y.shape}")

# Train Meta-Classifier
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
from sklearn.linear_model import LogisticRegression
meta_clf = MultiOutputClassifier(LogisticRegression(random_state=42))
meta_clf.fit(X_train, y_train)

# Save Meta-Classifier
joblib.dump(meta_clf, "meta_classifier.pkl")

Streaming output truncated to the last 5000 lines.

image 1/1 /tmp/crop.jpg: 480x640 (no detections), 10.5ms
Speed: 2.5ms preprocess, 10.5ms inference, 0.8ms postprocess per image at shape (1, 3, 480, 640)

image 1/1 /tmp/crop.jpg: 288x640 1 Fishing trawler, 12.3ms
Speed: 1.8ms preprocess, 12.3ms inference, 2.7ms postprocess per image at shape (1, 3, 288, 640)

image 1/1 /content/Traditional-boats-of-Bangladesh-16/train/images/istockphoto-2152099505-612x612_jpg.rf.65040be2166cd5555c9a3594c5013abb.jpg: 640x640 3 Fishing trawlers, 9.1ms
Speed: 2.7ms preprocess, 9.1ms inference, 1.8ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 /tmp/crop.jpg: 416x640 1 Fishing trawler, 10.5ms
Speed: 2.2ms preprocess, 10.5ms inference, 2.8ms postprocess per image at shape (1, 3, 416, 640)

image 1/1 /tmp/crop.jpg: 320x640 1 Fishing trawler, 11.6ms
Speed: 1.9ms preprocess, 11.6ms inference, 2.6ms postprocess per image at shape (1, 3, 320, 640)

image 1/1 /tmp/crop.jpg: 512x640 1 Fishing trawl

lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the doc

['meta_classifier.pkl']

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def visualize_segmentation_with_geometry(image_path, seg_model):
    # Run YOLO segmentation
    results = seg_model.predict(source=str(image_path), save=False)
    result = results[0]
    masks = result.masks

    if masks is None:
        print("No masks found.")
        return

    img = cv2.imread(str(image_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    geometry_details = []

    for idx, mask in enumerate(masks.data.cpu().numpy()):
        binary_mask = (mask * 255).astype(np.uint8)

        # Find contours
        contours, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours:
            continue

        cnt = max(contours, key=cv2.contourArea)

        # Calculate area
        area = cv2.contourArea(cnt)

        # Bounding box
        x, y, w, h = cv2.boundingRect(cnt)
        elongation = w / h if h > 0 else 0

        # Draw mask contour and box on image
        cv2.drawContours(img, [cnt], -1, (0, 255, 0), 2)
        cv2.rectangle(img, (x, y), (x + w, y + h), (255, 0, 0), 2)
        cv2.putText(img, f"Boat {idx+1}: A={area:.1f}, E={elongation:.2f}",
                    (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1)

        geometry_details.append({
            "Boat": idx + 1,
            "Area": area,
            "Elongation": elongation
        })

    # Show image with annotations
    plt.figure(figsize=(10, 10))
    plt.imshow(img)
    plt.axis("off")
    plt.title("Segmentation with Geometric Details")
    plt.show()

    # Print geometry details
    for detail in geometry_details:
        print(f"Boat {detail['Boat']}: Area = {detail['Area']:.1f}, Elongation = {detail['Elongation']:.2f}")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import cv2
import torch

def visualize_processing(image_path, label_path=None):
    # Load image
    image = Image.open(image_path).convert('RGB')
    img_array = np.array(image)

    # Initialize plot
    plt.figure(figsize=(15, 10))

    # Step 1: Display original image
    plt.subplot(2, 3, 1)
    plt.imshow(img_array)
    plt.title('Original Image')
    plt.axis('off')

    # Step 2: YOLOv8 Detection
    yolo_feat, bboxes = extract_yolo_features(image_path)
    img_with_boxes = img_array.copy()
    for box in bboxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        cls = int(box.cls[0])
        conf = float(box.conf[0])
        cv2.rectangle(img_with_boxes, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(img_with_boxes, f'Class {cls} ({conf:.2f})', (x1, y1-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
    plt.subplot(2, 3, 2)
    plt.imshow(img_with_boxes)
    plt.title(f'YOLO Detection\nFeatures: {yolo_feat.tolist()}')
    plt.axis('off')

    # Step 3 & 4: ResNet and YOLO Segmentation on Crops
    resnet_probs_list = []
    seg_features_list = []
    crop_images = []
    for i, box in enumerate(bboxes[:5]):  # Limit to 3 crops
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(image.size[0], x2), min(image.size[1], y2)
        if x2 <= x1 or y2 <= y1:
            continue
        crop = image.crop((x1, y1, x2, y2))
        crop_images.append(np.array(crop))

        # ResNet Probabilities
        resnet_probs = extract_resnet_probs_crop(image, box, img_size=128)
        resnet_probs_list.append(resnet_probs)

        # YOLO Segmentation
        seg_features = extract_yolo_seg_features_crop(image, box)
        seg_features_list.append(seg_features)

        # Display Crop and Features
        plt.subplot(2, 5, 5 + i)
        plt.imshow(crop_images[-1])
        plt.title(f'Crop {i+1}\nResNet Probs: {resnet_probs.round(2).tolist()}\nSeg: {seg_features.round(2).tolist()}')
        plt.axis('off')

    # Aggregate features
    resnet_feat = np.mean(resnet_probs_list, axis=0) if resnet_probs_list else np.zeros(NUM_CLASSES)
    seg_feat = np.mean(seg_features_list, axis=0) if seg_features_list else np.array([0, 0])
    meta_feat = np.concatenate([yolo_feat, resnet_feat, seg_feat])

    # Step 5: Meta-Classifier Prediction
    pred = meta_clf.predict(meta_feat.reshape(1, -1))[0]
    pred_classes = [i for i, val in enumerate(pred) if val == 1]

    # Step 6: Ground Truth (if available)
    gt_classes = []
    if label_path and label_path.exists():
        label_vec = get_multilabel_vector(label_path)
        gt_classes = [i for i, val in enumerate(label_vec) if val == 1]

    # Display Final Result
    plt.subplot(2, 3, 6)
    plt.text(0.1, 0.5, f'Meta-Classifier Prediction: {pred_classes}\nGround Truth: {gt_classes}\nMeta-Features: {meta_feat.round(2).tolist()}',
             fontsize=10, verticalalignment='center')
    plt.title('Final Result')
    plt.axis('off')

    plt.tight_layout()
    plt.show()

# Example Usage
from pathlib import Path

# Define paths
image_dir = DATASET_DIR / 'train/images'
label_dir = DATASET_DIR / 'train/labels'

# Get list of image paths
image_paths = sorted(image_dir.glob('*.jpg'))[120:150]

# Loop over each image and visualize
for img_path in image_paths:
    label_path = label_dir / (img_path.stem + '.txt')
    visualize_processing(img_path, label_path)


NameError: name 'DATASET_DIR' is not defined

In [ ]:
# Evaluate on Validation Set
y_val_pred = meta_clf.predict(X_val)
print("\n🔍 Validation Classification Report:")
print(classification_report(y_val, y_val_pred, zero_division=0))
print(f"🎯 Subset Accuracy: {accuracy_score(y_val, y_val_pred):.4f}")
print(f"🎯 Hamming Loss: {hamming_loss(y_val, y_val_pred):.4f}")
print(f"🎯 Micro F1 Score: {f1_score(y_val, y_val_pred, average='micro'):.4f}")
print(f"🎯 Macro F1 Score: {f1_score(y_val, y_val_pred, average='macro'):.4f}")

# Confusion Matrices
class_names = ['Nouka', 'Dingi', 'Bajra', 'Jali', 'Kosa', 'Others']
cm = multilabel_confusion_matrix(y_val, y_val_pred)
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for i, ax in enumerate(axes.flat):
    sns.heatmap(cm[i], annot=True, fmt='d', cmap='Blues', ax=ax)
    ax.set_title(f'{class_names[i]} (Class {i}) Confusion Matrix')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
plt.tight_layout()
plt.show()


# ROC Curves
if hasattr(meta_clf, "predict_proba"):
    y_score = np.array([est.predict_proba(X_val)[:, 1] for est in meta_clf.estimators_]).T
    fig = plt.figure(figsize=(10, 8))
    for i in range(NUM_CLASSES):
        fpr, tpr, _ = roc_curve(y_val[:, i], y_score[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f'Class {i} (AUC = {roc_auc:.2f})')
    plt.plot([0, 1], [0, 1], 'k--')
    plt.title('ROC Curves per Class (Validation Set)')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.legend(loc='lower right')
    plt.grid(True)
    plt.show()
else:
    print("⚠️ predict_proba not available.")

# Evaluate on Test Set
X_test, y_test, test_image_paths = [], [], []
for img_path in sorted(TEST_IMAGES_DIR.glob("*.jpg")):
    label_path = TEST_LABELS_DIR / (img_path.stem + ".txt")
    if not label_path.exists():
        print(f"Skipping {img_path.name}: Missing label file")
        continue
    label_vec = get_multilabel_vector(label_path)
    feat_vec = build_meta_feature(img_path)
    if feat_vec.size == 0:
        print(f"Skipping {img_path.name}: Empty feature vector")
        continue
    X_test.append(feat_vec)
    y_test.append(label_vec)
    test_image_paths.append(img_path)

X_test, y_test = np.array(X_test), np.array(y_test)
print(f"✅ Test samples: {len(X_test)}")

y_test_pred = meta_clf.predict(X_test)
print("\n🔍 Test Classification Report:")
print(classification_report(y_test, y_test_pred, zero_division=0))
print(f"🎯 Subset Accuracy: {accuracy_score(y_test, y_test_pred):.4f}")
print(f"🎯 Hamming Loss: {hamming_loss(y_test, y_test_pred):.4f}")
print(f"🎯 Micro F1 Score: {f1_score(y_test, y_test_pred, average='micro'):.4f}")
print(f"🎯 Macro F1 Score: {f1_score(y_test, y_test_pred, average='macro'):.4f}")
from sklearn.metrics import multilabel_confusion_matrix, roc_curve, auc
import seaborn as sns
import matplotlib.pyplot as plt

# Define class names
class_names = ['Nouka', 'Dingi', 'Bajra', 'Jali', 'Kosa', 'Others']

# Confusion Matrices
cm = multilabel_confusion_matrix(y_val, y_val_pred)
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for i, ax in enumerate(axes.flat):
    sns.heatmap(cm[i], annot=True, fmt='d', cmap='Blues', ax=ax)
    ax.set_title(f'{class_names[i]} (Class {i}) Confusion Matrix')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
plt.tight_layout()
plt.show()

# ROC Curves
if hasattr(meta_clf, "predict_proba"):
    # Get probability scores for each estimator
    y_score = np.array([est.predict_proba(X_val)[:, 1] for est in meta_clf.estimators_]).T

    # Plot ROC Curve per class
    fig = plt.figure(figsize=(10, 8))
    for i in range(len(class_names)):
        fpr, tpr, _ = roc_curve(y_val[:, i], y_score[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f'{class_names[i]} (AUC = {roc_auc:.2f})')

    plt.plot([0, 1], [0, 1], 'k--')  # Diagonal
    plt.title('ROC Curves per Class (Validation Set)')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.legend(loc='lower right')
    plt.grid(True)
    plt.show()
else:
    print("⚠️ predict_proba not available on the meta-classifier.")



NameError: name 'meta_clf' is not defined

In [ ]:
print("X_val shape:", X_val.shape)
print("X_val sample:", X_val[0])
print("y_val sample:", y_val[0])


X_val shape: (200, 16)
X_val sample: [          4     0.56815           0           1           0           0           3           0   0.0020323     0.10119  0.00097056     0.01378     0.83639   0.0090697       37503      1.7274]
y_val sample: [          0           1           0           0           1           0]


In [ ]:
# Directory for new/validation images
NEW_IMAGES_DIR = Path("/content/Traditional-boats-of-Bangladesh-16/valid/images")

# --- Prediction Functions ---

def predict_yolo(image_path):
    results = yolo_model.predict(source=str(image_path), save=False)
    detected = set()
    for result in results:
        for box in result.boxes:
            cls = int(box.cls[0])
            detected.add(cls)
    return sorted(list(detected))


def extract_resnet_probs(image_path):
    # For whole image inference (optional, not crop-based)
    image = Image.open(image_path).convert('RGB')
    tensor = transform(image).unsqueeze(0).to(device)
    resnet_model.eval()
    with torch.no_grad():
        logits = resnet_model(tensor).cpu().squeeze(0).numpy()
        probs = 1 / (1 + np.exp(-logits))
    return probs


def predict_resnet(image_path, threshold=0.3):
    probs = extract_resnet_probs(image_path)
    classes = [i for i, p in enumerate(probs) if p >= threshold]
    return classes, probs


def predict_meta(image_path):
    feat = build_meta_feature(image_path).reshape(1, -1)
    pred = meta_clf.predict(feat)[0]
    return [i for i, val in enumerate(pred) if val == 1]


# --- Run Inference on Validation Images ---
new_image_paths = sorted(NEW_IMAGES_DIR.glob("*.jpg"))[:5]  # LIMIT TO FIRST 5

for img_path in new_image_paths:
    yolo_classes = predict_yolo(img_path)
    resnet_classes, resnet_probs = predict_resnet(img_path)
    meta_classes = predict_meta(img_path)

    # YOLOv8 Detection Visualization
    results = yolo_model.predict(source=str(img_path), save=False)
    im_rgb = results[0].plot()[..., ::-1]

    # Plot the detection and classification results
    plt.figure(figsize=(10, 10))
    plt.imshow(im_rgb)
    plt.axis("off")
    plt.title(
        f"{img_path.name}\n"
        f"YOLO: {', '.join([f'Class {c}' for c in yolo_classes]) or 'None'}\n"
        f"ResNet: {', '.join([f'Class {c}' for c in resnet_classes]) or 'None'}\n"
        f"Meta: {', '.join([f'Class {c}' for c in meta_classes]) or 'None'}"
    )
    plt.show()

    print(f"🔍 ResNet probs: {np.round(resnet_probs, 3)}\n")

    # Segmentation-based geometry visualization (if defined)
    visualize_segmentation_with_geometry(img_path, seg_model)


Output hidden; open in https://colab.research.google.com to view.

In [ ]:
# Save Models for Download
from google.colab import files
files.download("resnet18_multilabel.pt")
files.download("meta_classifier.pkl")
files.download("/content/best.pt")
files.download("/content/best.pt")